# Tugas 7 : Klasifikasi Berita Metode Logistic Regression dengan reduksi dimensi

**Nama : Achmad Baharuddin Akbar**

**NIM  : 210411100001**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/My Drive/PPW-A/report/tugas-ppw/hasil_preprocesing.csv")
df.head()

,judul,tanggal,isi,kategori,cleansing,case_folding,tokenize,Filtering/stopword removal
0,Gejala Sifilis pada Wanita Berdasarkan Tahapan...,"Rabu, 16 Okt 2024 21:00 WIB",Jakarta - Sifilis atau penyakit raja singa ter...,Kesehatan,Jakarta Sifilis atau penyakit raja singa term...,jakarta sifilis atau penyakit raja singa term...,"['jakarta', 'sifilis', 'atau', 'penyakit', 'ra...",jakarta sifilis penyakit raja singa infeksi me...
1,Puncak Nafsu Pria Ada di Umur Berapa? Studi Bi...,"Rabu, 16 Okt 2024 20:02 WIB",Jakarta - Sebagian orang pasti sudah mengenal ...,Kesehatan,Jakarta Sebagian orang pasti sudah mengenal i...,jakarta sebagian orang pasti sudah mengenal i...,"['jakarta', 'sebagian', 'orang', 'pasti', 'sud...",jakarta orang mengenal istilah puncak seksual ...
2,"Gejala Kanker Mulut yang Kerap Tak Disadari, T...","Rabu, 16 Okt 2024 18:02 WIB",Jakarta - Setiap orang mungkin pernah mengalam...,Kesehatan,Jakarta Setiap orang mungkin pernah mengalami...,jakarta setiap orang mungkin pernah mengalami...,"['jakarta', 'setiap', 'orang', 'mungkin', 'per...",jakarta orang mengalami sariawan luka mulut me...
3,Kapan Waktu yang Tepat untuk Minum Air Rebusan...,"Rabu, 16 Okt 2024 17:31 WIB",Jakarta - Air rebusan serai dikenal sebagai sa...,Kesehatan,Jakarta Air rebusan serai dikenal sebagai sal...,jakarta air rebusan serai dikenal sebagai sal...,"['jakarta', 'air', 'rebusan', 'serai', 'dikena...",jakarta air rebusan serai dikenal salah ramuan...
4,Viral Hanni NewJeans Bicara soal Bullying di T...,"Rabu, 16 Okt 2024 16:34 WIB","Jakarta - Viral momen member NewJeans, Hanni, ...",Kesehatan,Jakarta Viral momen member NewJeans Hanni ber...,jakarta viral momen member newjeans hanni ber...,"['jakarta', 'viral', 'momen', 'member', 'newje...",jakarta viral momen member newjeans hanni berb...


## **Label Encoder**

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Transformasi data kategorik
label_encoder = LabelEncoder()
df.loc[:, 'kategori_encoded'] = label_encoder.fit_transform(df['kategori'])

# Menampilkan nilai sebelum dan sesudah konversi
print("\nNilai sebelum dan sesudah konversi:")
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

print("\nDataFrame setelah Label Encoding:")
df.head()


Nilai sebelum dan sesudah konversi:
{'Kesehatan': 0, 'Kuliner': 1}

DataFrame setelah Label Encoding:


,judul,tanggal,isi,kategori,cleansing,case_folding,tokenize,Filtering/stopword removal,kategori_encoded
0,Gejala Sifilis pada Wanita Berdasarkan Tahapan...,"Rabu, 16 Okt 2024 21:00 WIB",Jakarta - Sifilis atau penyakit raja singa ter...,Kesehatan,Jakarta Sifilis atau penyakit raja singa term...,jakarta sifilis atau penyakit raja singa term...,"['jakarta', 'sifilis', 'atau', 'penyakit', 'ra...",jakarta sifilis penyakit raja singa infeksi me...,0
1,Puncak Nafsu Pria Ada di Umur Berapa? Studi Bi...,"Rabu, 16 Okt 2024 20:02 WIB",Jakarta - Sebagian orang pasti sudah mengenal ...,Kesehatan,Jakarta Sebagian orang pasti sudah mengenal i...,jakarta sebagian orang pasti sudah mengenal i...,"['jakarta', 'sebagian', 'orang', 'pasti', 'sud...",jakarta orang mengenal istilah puncak seksual ...,0
2,"Gejala Kanker Mulut yang Kerap Tak Disadari, T...","Rabu, 16 Okt 2024 18:02 WIB",Jakarta - Setiap orang mungkin pernah mengalam...,Kesehatan,Jakarta Setiap orang mungkin pernah mengalami...,jakarta setiap orang mungkin pernah mengalami...,"['jakarta', 'setiap', 'orang', 'mungkin', 'per...",jakarta orang mengalami sariawan luka mulut me...,0
3,Kapan Waktu yang Tepat untuk Minum Air Rebusan...,"Rabu, 16 Okt 2024 17:31 WIB",Jakarta - Air rebusan serai dikenal sebagai sa...,Kesehatan,Jakarta Air rebusan serai dikenal sebagai sal...,jakarta air rebusan serai dikenal sebagai sal...,"['jakarta', 'air', 'rebusan', 'serai', 'dikena...",jakarta air rebusan serai dikenal salah ramuan...,0
4,Viral Hanni NewJeans Bicara soal Bullying di T...,"Rabu, 16 Okt 2024 16:34 WIB","Jakarta - Viral momen member NewJeans, Hanni, ...",Kesehatan,Jakarta Viral momen member NewJeans Hanni ber...,jakarta viral momen member newjeans hanni ber...,"['jakarta', 'viral', 'momen', 'member', 'newje...",jakarta viral momen member newjeans hanni berb...,0


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# Memisahkan fitur dan label
x = df['Filtering/stopword removal']
y = df['kategori_encoded']

## **Split Data**

In [ ]:
# Membagi data menjadi data latih dan data uji
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## **TF-IDF (Term Frequency-Inverse Document Frequency)**

In [ ]:
# Tahap 1: TF-IDF
tfidf_vectorizer = TfidfVectorizer()
x_train_tfidf = tfidf_vectorizer.fit_transform(x_train)
x_test_tfidf = tfidf_vectorizer.transform(x_test)

# Mengonversi hasil TF-IDF menjadi DataFrame
df_train_tfidf = pd.DataFrame(x_train_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
df_test_tfidf = pd.DataFrame(x_test_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
print("Hasil TF-IDF (Data Latih):")
df_train_tfidf

Hasil TF-IDF (Data Latih):


,abai,abang,abc,abdalla,abdullah,abis,abnormal,acar,acara,aceh,...,yorkpresbyteriancolumbia,youtuber,yu,yudhistira,yun,yuni,zaitun,zaman,zat,zonk
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.039679,0.0,0.0,0.0,0.0,0.0,0.039679,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.078811,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
76,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
77,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
78,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0


## **Reduksi Dimensi (SVD)**

In [ ]:
# Tahap 2: SVD (Reduksi Dimensi)
svd = TruncatedSVD(n_components=100, random_state=42)  # Sesuaikan n_components dengan kebutuhan
x_train_svd = svd.fit_transform(x_train_tfidf)
x_test_svd = svd.transform(x_test_tfidf)

# Mengonversi hasil SVD menjadi DataFrame
df_train_svd = pd.DataFrame(x_train_svd)
df_test_svd = pd.DataFrame(x_test_svd)
print("\nHasil SVD (Data Latih):")
df_train_svd


Hasil SVD (Data Latih):


,0,1,2,3,4,5,6,7,8,9,...,70,71,72,73,74,75,76,77,78,79
0,0.146835,0.098446,0.036118,-0.007853,0.030171,0.030573,0.101881,-0.130841,0.317646,-0.017058,...,-0.006040,-0.031587,0.028166,0.017332,0.019683,-0.014127,0.008719,0.002296,0.000250,0.005757
1,0.103576,0.088657,-0.036154,-0.024804,-0.020516,0.022535,-0.036073,-0.078213,0.064766,0.041046,...,0.015789,0.027988,-0.022725,-0.013168,0.002772,0.004086,-0.006429,0.001925,-0.001272,0.001696
2,0.128020,0.021011,-0.291435,-0.106443,-0.062714,0.118524,-0.002286,0.164205,-0.056252,-0.135308,...,0.031615,-0.020913,0.002676,0.014624,0.009029,-0.003808,-0.000068,0.000172,0.003350,0.000546
3,0.180694,0.041392,-0.354137,-0.126201,-0.100935,0.140029,0.005914,0.170070,-0.010579,0.011061,...,-0.037820,0.067192,0.017925,-0.009966,-0.002835,-0.003059,0.002702,-0.000194,-0.005255,0.001155
4,0.231558,0.144722,-0.009766,-0.023885,-0.106261,-0.084053,-0.107354,-0.185719,0.095399,0.050984,...,-0.047783,0.020885,-0.031002,-0.000453,0.009014,-0.000789,0.012800,0.004507,-0.000309,0.002150
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0.112538,0.066161,0.005971,-0.001866,0.008656,0.017022,-0.014257,-0.102787,0.148968,-0.008918,...,-0.002196,0.007912,-0.017573,-0.011353,0.010643,-0.002733,0.005110,0.006552,0.000897,0.002267
76,0.209757,0.278506,0.255440,0.095555,0.335217,0.732613,-0.173662,-0.040365,-0.119820,0.035642,...,-0.028113,-0.021989,0.006352,-0.002051,-0.000453,0.003172,0.001547,0.004510,-0.015086,0.300916
77,0.234076,-0.137178,-0.051662,-0.107811,0.059411,0.064492,0.515711,-0.332924,-0.419539,0.095469,...,0.030637,-0.008129,0.038226,-0.014870,-0.444102,0.126856,0.043935,0.002291,-0.001539,-0.000216
78,0.133118,0.087723,0.001356,0.023964,-0.000093,-0.025817,0.131952,-0.096307,0.277163,0.006594,...,-0.007083,0.025409,-0.016980,-0.001284,-0.010862,0.004845,0.020646,-0.014121,-0.000254,-0.001365


## **Model Logistic Regression**

In [ ]:
# Tahap 3: Klasifikasi dengan Logistic Regression
clf = LogisticRegression(random_state=42)
clf.fit(x_train_svd, y_train)
y_pred = clf.predict(x_test_svd)

In [ ]:
# Evaluasi Model
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.85
              precision    recall  f1-score   support

           0       0.91      0.83      0.87        12
           1       0.78      0.88      0.82         8

    accuracy                           0.85        20
   macro avg       0.84      0.85      0.85        20
weighted avg       0.86      0.85      0.85        20



In [ ]:
# Simpan TF-IDF vectorizer dan model
import pickle

with open('svd_tfidf.pkl', 'wb') as tfidf_file:
    pickle.dump(x_train_svd, tfidf_file)

with open('logistic_regression_model.pkl', 'wb') as model_file:
    pickle.dump(clf, model_file)

## **Pengujian**

In [ ]:
import joblib
from sklearn.pipeline import Pipeline

# Misalnya, Anda telah membuat pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('svd', TruncatedSVD(n_components=100)),
    ('clf', LogisticRegression())
])

# Setelah melatih model
pipeline.fit(x_train, y_train)

# Menyimpan pipeline
joblib.dump(pipeline, 'svd_tfidf_pipeline.pkl')

['svd_tfidf_pipeline.pkl']

In [ ]:
# Teks baru yang ingin diprediksi
new_text_1 = ['''
Jakarta - Sejumlah menteri, termasuk Menteri Kesehatan RI Budi Gunadi Sadikin menghadap Presiden Terpilih Prabowo Subianto di Kertanegara, Jakarta Selatan, Senin (14/10/2024). Menkes mengaku membahas banyak hal, khususnya terkait persoalan di sektor kesehatan. Salah satu yang dibahas terkait masalah jumlah dokter, termasuk dokter spesialis di Indonesia. "Saya diajak bicara dengan beliau mengenai masalah kesehatan, jumlah dokter harus cukup, jumlah dokter spesialis harus cukup, bagaimana pendidikan dokter dan dokter spesialis diperbanyak,katanya di Jalan Kertanegara, Jakarta Selatan, Senin (14/10/2024). "Kalau masyarakat masih kekurangan bisa nggak belajarnya tak hanya di dalam negeri saja tetapi juga di luar negeri untuk mempercepat," lanjutnya lagi. Selain jumlah dokter, Menkes juga membicarakan terkait penyakit yang beredar di Indonesia, seperti malaria hingga tuberkulosis. Juga, Presiden Terpilih Prabowo Subianto menginginkan masyarakat yang sehat melalui peningkatan pemeriksaan kesehatan di Indonesia. "Beliau juga ingin pemeriksaan kesehatan terus ditingkatkan agar jangan sampai sakit masyarakatnya, kalau bisa kita jaga supaya sehat terus," lanjutnya lagi. "Beliau juga pesankan ada penyakit-penyakit yang menurut beliau sudah bisa diatasi di Indonesia, seperti tuberkulosis sama malaria, itu harusnya bisa cepat dieliminasi dicari vaksinnya supaya masyarakat kita bisa cepat hilang dari penyakit ini," katanya lagi. Ketika ditanya apakah dirinya bakal menjadi Menkes kembali, Budi menjawab nanti bakal diumumkan langsung oleh Presiden Terpilih. Hingga saat ini, Menkes Budi belum memberitahu jabatan apa yang akan ditawarkan oleh Presiden Terpilih Prabowo Subianto. (suc/suc)
''']

new_text_2 = ['''
Jakarta - Mencuri kue milik rekan kerjanya dari kulkas, karyawan ini kena batunya. Karena kue tersebut ternyata mengandung ganja. Alhasil, mereka dilarikan ke rumah sakit. Tidak ada yang lebih menyebalkan di lingkungan kantor daripada rekan kerja yang terus-menerus memakan makanan milik karyawan lainnya dari kulkas yang ada di dapur bersama. Mereka tahu itu bukan milik mereka, karenanya mereka mengambil secara diam-diam. Hal ini sangat menyebalkan apalagi ketika si pemilik makanan itu sudah menanti-nantikan untuk menikmatinya. Berharap si pencuri makanan itu kena batunya. Seperti yang terjadi di sebuah perkantoran. Di mana seorang karyawan mencuri kue milik rekan kerjanya yang ada di kulkas secara diam-diam. Karyawan Curi Kue Milik Rekannya dan Ternyata Mengandung Ganja. Ilustrasi pai. Foto: iStock Tanpa sadar bahwa kue tersebut mengandung THC atau senyawa yang terkandung pada ganja. Hal ini diceritakan oleh Fola, pemilik Sades Dulces, tempat dia menjual aneka kue, lapor Upworthy (03/10/24). Lewat media sosialnya, ia menceritakan bahwa ia mendapat panggilan telepon dari bisnis lokal tentang kue yang dikonsumsi oleh karyawannya. Fola mengungkapkan bahwa di luar Sades Dulces miliknya, ia menjual pai yang mengandung THC, yaitu jenis pai yang dikirim ke perusahaan pelanggan. Negara tempat hal ini terjadi telah melegalkan penjualan dan konsumsi ganja, termasuk produk apapun yang berasal dari minyak THC, sehingga seseorang yang memesan pai ini bukanlah hal yang aneh. Dalam kasus ini, Fola diminta untuk mengantarkan pai tersebut agar pelanggan dapat membawa pulang pai untuk akhir pekan. Sayangnya, si pelanggan lupa membawa pai yang disimpan di kulkas kantor. Ilustrasi ganja. Foto: iStock Saat itulah, teman kantornya yang hobi mencuri makanan rekan kerja mengambil kue tersebut. Tak hanya dimakan sendiri, ia juga membagikan ke 8 rekan kerja lainnya. Tak ada satupun yang mengetahui bahwa pai tersebut mengandung ganja. Alhasil, mereka yang memakan pai itu langsung jatuh sakit, sehingga atasan mereka harus memanggil ambulans. Karena itulah, atasan mereka menghubungi Fola sebagai penjual pai. Fola baru memberi tahu bahwa kue yang dimakan tersebut mengandung THC. Namun, Fola menolak memberikan nama pelanggannya karena khawatir mereka akan mendapat masalah. Akhirnya Fola menelepon pelanggan tersebut untuk memberitahu dia tentang 9 rekan kerjanya yang dilarikan ke rumah sakit, setelah makan pai mengandung THC miliknya. "Pelanggan saya itu tidak merasa kasihan, justru kenapa rekannya memakan kue tanpa izin? Menurutnya ini sama saja pencurian. Dan itulah akibatnya, beruntung mereka baik-baik saja setelah dari rumah sakit," tutup Fola. (raf/odi)
''']

# Gabungkan kedua teks menjadi satu list
new_text = new_text_1 + new_text_2

# Melakukan prediksi dengan pipeline
predictions = pipeline.predict(new_text)

# Hasil prediksi
predicted_categories = ["Kesehatan" if prediction == 0 else "Kuliner" for prediction in predictions]

# Output hasil prediksi untuk masing-masing teks
for i, category in enumerate(predicted_categories):
    print(f"Prediksi untuk teks {i+1}: {category}")

Prediksi untuk teks 1: Kesehatan
Prediksi untuk teks 2: Kuliner
